# BERT fake-vs-real classifier on HF-vs-HR (PolitiFact++ & GossipCop++)

Adapts `fake-vs-real-news-detection-bert-acc-100.ipynb` (`bert-base-uncased`), pointed at the
LIFE **human-written** subset. **Task = fake-vs-real among human news:** HF (human_fake) =
**0 (fake)**, HR (human_true) = **1 (real)** — the *same cut* as `run_life_lstm.py` (§7n), so this
is a direct **BERT-vs-LSTM head-to-head**.

- The source notebook is TF/Keras, but current Colab `transformers` dropped TF support (Keras 3),
  so this runs BERT in **PyTorch** (same model/technique; the stack that ran the RoBERTa track).
- Same pipeline: clean (lowercase/stopwords/punct) → BERT tokenize → fine-tune 5 epochs,
  AdamW 1e-5, stratified 90/10→90/10 split, Fake=0/Real=1.
- **The source notebook's "100%" is a source-leakage artifact** (ISOT real news is all
  Reuters-formatted). Expect **honest, lower** numbers on LIFE data.
- **Caveats:** PolitiFact++ HF/HR is only **291 articles** (~29 test) → noisy; BERT may still
  beat the LSTM's collapse there. GossipCop++ (**12,252**) is the longer run (~10–15 min on GPU,
  all articles by default). Both are **1:2 fake:real** → watch **fake(HF=0) recall** + the
  confusion matrix, not just accuracy.
- **GPU required** (Runtime → Change runtime type → GPU).

In [11]:
!pip install -q transformers  # torch is preinstalled on Colab

In [12]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU')

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path (and its `import run_life_lstm`) resolve
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

PolitiFact++ found: True
GossipCop++  found: True


## Run the classifier

Each cell prints the class balance, per-epoch train/val accuracy, then a test
`classification_report` + confusion matrix. PolitiFact++ is quick; GossipCop++ (all ~12k
articles, 5 epochs) takes ~10–15 min on a GPU.

In [17]:
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

config.json: 100% 570/570 [00:00<00:00, 3.32MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 281kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.74MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.99MB/s]
[PolitiFact++] 271 articles | fake HF(0)=93 real HR(1)=178 | device=cuda

model.safetensors: downloading bytes:  34% 151M/440M [00:01<00:01, 230MB/s, 9.37MB/s  ] 
model.safetensors: downloading bytes:  79% 350M/440M [00:01<00:00, 404MB/s, 27.9MB/s  ]
model.safetensors: reconstructing file:  30% 134M/440M [00:01<00:03, 91.1MB/s, 6.55MB/s  ]
model.safetensors: downloading bytes:  92% 405M/440M [00:01<00:00, 298MB/s, 37.0MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 184MB/s, 38.1MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 195MB/s, 41.3MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5221.46it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                   

In [18]:
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

[GossipCop++] 11081 articles | fake HF(0)=3590 real HR(1)=7491 | device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 6149.73it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

## Notes
- **Task = fake-vs-real among human news** (HF=0 vs HR=1) — the same cut as the LSTM (§7n), for a
  BERT-vs-LSTM comparison. It imports `FILE_LABELS` + `load_dataframe` from `run_life_lstm.py`, so
  **sync that file to Drive too** (both live in `lstm_real_vs_fake_code/`).
- Follows the source notebook (bert-base-uncased, lr 1e-5, 5 epochs, stratified 90/10→90/10)
  but in **PyTorch** (its TF path is unavailable on current Colab), with num_labels=2 +
  cross-entropy, subsample capped at dataset size, and `--max_length 256`.
- GossipCop++ uses **all** articles by default (`--sample_size 0`); pass e.g. `--sample_size 1000`
  for the source notebook's fast subsample behavior. No model checkpoints saved.